# AthleteLoad: 04. Model Explainability & SHAP Analysis

This notebook uses TreeSHAP to interpret the decisions of our XGBoost model. We examine global feature importances, feature impact beeswarm plots, and local waterfall explanations for individual athlete-days.

In [ ]:
import os
import sys
# Ensure parent directory is in python path
if os.path.abspath('..') not in sys.path:
    sys.path.insert(0, os.path.abspath('..'))

def get_path(rel_path):
    if os.path.exists(rel_path):
        return rel_path
    elif os.path.exists(os.path.join('..', rel_path)):
        return os.path.join('..', rel_path)
    return rel_path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from src.utils import setup_plotting_style, PALETTE
from src.train import run_full_training_pipeline
setup_plotting_style()

artifacts = run_full_training_pipeline(data_path=get_path('data/processed/player_daily_features.csv'), seed=42)
xgb_model = artifacts['models']['XGBoost']
feature_names = artifacts['feature_cols']
X_test = artifacts['X_test']
y_test = artifacts['y_test']

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer(X_test)
shap_values.feature_names = feature_names
print('TreeSHAP values computed successfully on test partition.')

## 1. Global TreeSHAP Summary (Beeswarm Plot)

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, feature_names=feature_names, max_display=15, show=False)
plt.title('Global TreeSHAP Summary Plot (Top 15 Workload & Wellness Drivers)', fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## 2. Mean Absolute SHAP Importance Ranking

In [ ]:
mean_abs_shap = np.mean(np.abs(shap_values.values), axis=0)
top_indices = np.argsort(mean_abs_shap)[::-1][:15]
top_features = [feature_names[i] for i in top_indices]
top_values = mean_abs_shap[top_indices]

plt.figure(figsize=(9, 6))
sns.barplot(x=top_values, y=top_features, palette='mako', hue=top_features, legend=False)
plt.title('Global Feature Importance Ranking (Mean |SHAP Value|)', fontsize=12, fontweight='bold')
plt.xlabel('Mean Absolute SHAP Value', fontweight='bold')
plt.ylabel('Feature', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Individual Athlete-Day Case Studies (Waterfall Explanations)

In [ ]:
test_probs = xgb_model.predict_proba(X_test)[:, 1]
high_risk_idx = np.argmax(test_probs)
low_risk_idx = np.argmin(test_probs)

print(f'High-Risk Athlete-Day: Index {high_risk_idx} | Predicted Probability: {test_probs[high_risk_idx]:.1%}')
print(f'Low-Risk Athlete-Day:  Index {low_risk_idx} | Predicted Probability: {test_probs[low_risk_idx]:.1%}')

plt.figure(figsize=(9, 6))
shap.plots.waterfall(shap_values[high_risk_idx], max_display=10, show=False)
plt.title(f'Case Study: Elevated-Risk Athlete-Day (Risk = {test_probs[high_risk_idx]:.1%})', fontsize=11, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 6))
shap.plots.waterfall(shap_values[low_risk_idx], max_display=10, show=False)
plt.title(f'Case Study: Low-Risk Athlete-Day (Risk = {test_probs[low_risk_idx]:.1%})', fontsize=11, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()